#**CHINESE**

In [19]:
import json
import pandas as pd

# Read a .jsonl file: each line is one JSON object
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if line:                               # skip the empty lines
                rows.append(json.loads(line))
    return rows

dep = load_jsonl("depressed_500.jsonl")
con = load_jsonl("control_500.jsonl")
print("Depressed users loaded:", len(dep))
print("Control users loaded:", len(con))

# Peek at how one user's data is structured (first 2,000 characters)
print("\nStructure of one depressed user:")
print(json.dumps(dep[0], ensure_ascii=False, indent=2)[:2000])

Depressed users loaded: 500
Control users loaded: 500

Structure of one depressed user:
{
  "label": {
    "depressed": true,
    "symptoms": {
      "self_reported": true,
      "interest_loss": false,
      "pleasure_loss": false,
      "energy_loss": false,
      "sadness": true,
      "sympathetic_arousal": false,
      "concentration_problem": false,
      "panic": false,
      "appetite_problem": false,
      "insomnia": true,
      "anxious": false,
      "self_blame": false,
      "retardation": false,
      "suicidal_ideation": false,
      "weight_problem": false,
      "agitation": false,
      "hypersomnia": false
    }
  },
  "user": {
    "avatar_url": "https://tva3.sinaimg.cn/crop.0.0.512.512.180/005MWppRjw8fbawmcpu8xj30e80e8aal.jpg?KID=imgbed,tva&Expires=1611284231&ssig=fXqv8ymfB6",
    "cover_image_url": "https://tva1.sinaimg.cn/crop.0.0.640.640.640/9d44112bjw1f1xl1c10tuj20hs0hs0tw.jpg",
    "description": "爱演唱会的boy",
    "follow_count": 300,
    "followers_count": 110

In [20]:
# Turn "users with lists of posts" into a simple table: one row = one post, We DROP all user profile info (names, pictures) for privacy.

rows = []
for label_val, users in [(1, dep), (0, con)]:          # 1 = depressed users, 0 = control users
    for i, u in enumerate(users):
        user_id = f"{'d' if label_val == 1 else 'c'}_{i:03d}"
        symptoms = (u.get("label") or {}).get("symptoms") or {} # control users may have none
        for t in u.get("tweets", []):
            rows.append({
                "user_id": user_id,
                "label": label_val,
                "self_reported": symptoms.get("self_reported", False),
                "suicidal_ideation": symptoms.get("suicidal_ideation", False),
                "is_origin": t.get("is_origin"),
                "text": t.get("text", ""),
            })

posts = pd.DataFrame(rows)
print("Total posts (incl. reposts):", len(posts))
print("\nOwn posts vs reposts:\n", posts["is_origin"].value_counts())

# Keep only the user's own posts (not reposts)
posts = posts[posts["is_origin"] == True].copy()
posts["n_chars"] = posts["text"].astype(str).str.len()

print("\nOwn posts per label:\n", posts["label"].value_counts())
print("\nPosts per user (median):", posts.groupby("user_id").size().median())
print("\nPost length in characters, by label:")
print(posts.groupby("label")["n_chars"].describe())

# Keyword check: "抑郁" = depression
posts["has_keyword"] = posts["text"].str.contains("抑郁")
print("\nPosts containing '抑郁' (depression), by label:")
print(pd.crosstab(posts["label"], posts["has_keyword"]))

# How many users have at least one own post mentioning 抑郁?
users_kw = posts.groupby(["label", "user_id"])["has_keyword"].any().groupby("label").sum()
print("\nUsers with at least one '抑郁' post:\n", users_kw)

# What does the label look like for a control user?
print("\nControl user label example:", con[0].get("label"))

pd.set_option("display.max_colwidth", 150)
posts[(posts["label"] == 1) & posts["has_keyword"]][["user_id", "text"]].head(8)

Total posts (incl. reposts): 194985

Own posts vs reposts:
 is_origin
True     146590
False     48395
Name: count, dtype: int64

Own posts per label:
 label
1    78725
0    67865
Name: count, dtype: int64

Posts per user (median): 120.5

Post length in characters, by label:
         count        mean         std  min   25%    50%    75%      max
label                                                                   
0      67865.0  492.619156  554.191230  1.0  99.0  335.0  682.0  13129.0
1      78725.0  196.432086  306.488707  1.0  20.0   67.0  312.0  18449.0

Posts containing '抑郁' (depression), by label:
has_keyword  False  True 
label                    
0            67820     45
1            75472   3253

Users with at least one '抑郁' post:
 label
0     38
1    363
Name: has_keyword, dtype: int64

Control user label example: {'depressed': False}


,user_id,text
100,d_000,"唉<br />小日子一天比一天过的没意思<br />有些事可以做，但是不想做<br />有些事没办法，但是必须做<br />一天天的在焦虑、茫然、自己生自己气中度过<br />没意思啊没意思<br />我的抑郁症应该第二期了吧！<span class=""url-icon""><img alt=..."
200,d_000,之所以不能喜欢 是因为他不能保护我 他没有能给我那种做错什么有人兜着的感觉 他也没有给我犯错的机会<br />去别的地方 需要东西 我不敢搞 我害怕 我害怕没人帮我扛 后来想了下 就算没去别的地方也是一样 可能那个别的地方让我更加不熟悉罢了吧！<br />如果一个人曾经给了你很多胡作非为的...
720,d_002,狗狗会因为缺爱抑郁，人当然也会
778,d_003,抑郁的人基本都是善良的 坏人会让别人抑郁
779,d_003,本以为我已度过最抑郁的时期 日子会慢慢好起来 直到前两天被确诊为抑郁症 我才明白那段经历在我身上留下的伤痕有多深 就像山川留给地形 河流冲刷过那样 <br />我已经和抑郁合为一体了 甚至也做好了和它共度一生的准备 我不知道坚持吃药有没有用 也不知道停药以后会不会复发 但现在的我 好像除了配...
836,d_003,Emo人群人均抑郁 属实真实 👍🏻
1970,d_008,"<a href=""https://m.weibo.cn/search?containerid=231522type%3D1%26t%3D10%26q%3D%23%E4%BC%9A%E7%AC%91%E7%9A%84%E6%8A%91%E9%83%81%E7%97%87%E6%82%A3%E..."
2016,d_009,"<a href=""https://m.weibo.cn/search?containerid=231522type%3D1%26t%3D10%26q%3D%23%E6%8A%91%E9%83%81%E7%97%87%23&isnewpage=1&luicode=10000011&lfid=..."


In [21]:
# clean Weibo text, remove empty/duplicate posts

!pip install jieba -q
import re, html, warnings
import jieba
from bs4 import BeautifulSoup
warnings.filterwarnings("ignore")

# log (counts posts anx users, since labels are per user)
log = []
def log_step(name, data):
    log.append({
        "step": name,
        "posts": len(data),
        "posts_label_1": int((data["label"] == 1).sum()),
        "posts_label_0": int((data["label"] == 0).sum()),
        "users_label_1": data[data["label"] == 1]["user_id"].nunique(),
        "users_label_0": data[data["label"] == 0]["user_id"].nunique(),
    })

log_step("0. own posts (reposts removed)", posts)

def clean_weibo(t):
    soup = BeautifulSoup(str(t), "html.parser")
    for img in soup.find_all("img"):          # Weibo emojis are images - keep their text
        img.replace_with(img.get("alt", ""))
    for br in soup.find_all("br"):
        br.replace_with(" ")
    t = soup.get_text(" ")                    # remove all other HTML tags, keep the text
    t = html.unescape(t)                      # &amp; -> &
    t = re.sub(r"[\u200b\u200e\u200f\u202a-\u202e\u2066-\u2069]", "", t)  # invisible characters
    t = re.sub(r"https?://\S+|www\.\S+", "", t)            # links
    t = re.sub(r"O?网页链接|转发微博|分享图片", "", t)       # Weibo filler: "web link", "repost", "share photo"
    t = re.sub(r"//@[^:：]+[:：]", " ", t)                  # repost-chain markers "//@name:"
    t = re.sub(r"@[\w\-]+", "[USER]", t)                   # usernames
    t = re.sub(r"#[^#]{1,50}#", "", t)                     # Weibo hashtags look like #topic# - remove
    t = re.sub(r"\S*的微博视频", "", t)                     # remove "xxx's Weibo video" filler
    t = re.sub(r"我在[:：]\s*\S*", "", t)         # remove "I'm at: <place>" location tags
    t = re.sub(r"\s+", " ", t).strip()            # extra spaces
    return t

posts["text"] = posts["text"].apply(clean_weibo)

# remove empty posts
posts = posts[posts["text"] != ""]
log_step("1. clean text + drop empty", posts)

# remove duplicates
posts = posts.drop_duplicates(subset=["user_id", "text"])
log_step("2. drop duplicates", posts)

# count Chinese words with jieba
posts["n_words"] = posts["text"].apply(lambda t: len([w for w in jieba.lcut(t) if w.strip()]))
posts["n_chars"] = posts["text"].str.len()
posts["has_keyword"] = posts["text"].str.contains("抑郁")

print(pd.DataFrame(log))

print("\nPost length AFTER cleaning (words), by label:")
print(posts.groupby("label")["n_words"].describe())

print("\nUsers with at least one post of >= N words:")
for m in [15, 20, 30]:
    ok = posts[posts["n_words"] >= m].groupby("label")["user_id"].nunique()
    print(f"  >= {m} words ->  label 1: {ok.get(1,0)} users,  label 0: {ok.get(0,0)} users")

print("\nDepressed users with a '抑郁' post of >= 15 words (needed for option D):",
      posts[(posts["label"] == 1) & posts["has_keyword"] & (posts["n_words"] >= 15)]["user_id"].nunique())

posts[["user_id", "label", "text", "n_words"]].sample(8, random_state=42)

                             step   posts  posts_label_1  posts_label_0  \
0  0. own posts (reposts removed)  146590          78725          67865   
1      1. clean text + drop empty  143507          77336          66171   
2              2. drop duplicates  140709          76283          64426   

   users_label_1  users_label_0  
0            500            500  
1            500            500  
2            500            500  

Post length AFTER cleaning (words), by label:
         count       mean        std  min   25%   50%   75%     max
label                                                              
0      64426.0  34.592835  69.040547  1.0   9.0  18.0  37.0  2202.0
1      76283.0  30.410472  43.164296  1.0  10.0  18.0  37.0  1668.0

Users with at least one post of >= N words:
  >= 15 words ->  label 1: 500 users,  label 0: 500 users
  >= 20 words ->  label 1: 500 users,  label 0: 499 users
  >= 30 words ->  label 1: 498 users,  label 0: 488 users

Depressed users with a '

,user_id,label,text,n_words
118233,c_086,0,包租婆收租啦 [馋嘴] 福州·福州职业技术学院,12
59271,d_302,1,有些人看着挺正常，吃饭，睡觉，聊天，大笑 其实心里面早就死透了,19
126476,c_130,0,sonny angel真的好🉑️爱 [爱你],11
84066,d_415,1,别看我一手一个手机身上还躺着个手机 实际上我一边帮我爸刷视频一边帮我妈写文章 我的手机呢？躺在我身上 因为我没有手再拿一个手机了 [抓狂] 孩子好累 [抓狂],50
122676,c_110,0,美图秀秀里这个彩色铅笔画风好可爱❤️,9
100898,d_494,1,谁能陪我慢慢变老,6
148827,c_245,0,✨复盘6月观影记录👇🏻 ⭐纪录片 《如果国宝会说话3》 这部系列我推荐一万遍都不为过！ 除了可以了解国宝背后的历史， 里面的动画和文案都特别好！ 《寻味东莞》 《风味人间2》 《伊顿公学》 《亿万富翁的饕餮盛宴》 ⭐喜马拉雅 百家讲坛《诸子百家》未完7月继续 ⭐综艺电视剧 《创造营2020》...,93
118004,c_086,0,初三好累啊…希望我能坚持下来！ [兔子],12


In [22]:
import os

MIN_WORDS, MAX_WORDS = 15, 100   # 15 = same as Arabic/Thai; 100 = keep posts tweet-sized
N_PER_CLASS, SEED = 50, 42
SELECTION = "keyword"            # "keyword" = option D, "random" = option A  maybe will be changed after the meeting if needed

# Save post lengths for the visualization person (no text = small file)
posts[["user_id", "label", "n_words", "n_chars", "has_keyword"]].to_csv(
    "chinese_post_lengths.csv", index=False)

# 1. keep posts between 15 and 100 words

pool = posts[(posts["n_words"] >= MIN_WORDS) & (posts["n_words"] <= MAX_WORDS)].copy()
pool = pool[~pool["text"].str.endswith("全文")]
log_step(f"3. keep {MIN_WORDS}-{MAX_WORDS} words", pool)

# 2. which posts are allowed for depressed users?
if SELECTION == "keyword":
    pool_dep = pool[(pool["label"] == 1) & pool["has_keyword"]]
else:
    pool_dep = pool[pool["label"] == 1]
pool_con = pool[pool["label"] == 0]

# 3. pick ONE random post per user
one_dep = pool_dep.groupby("user_id").sample(n=1, random_state=SEED)
one_con = pool_con.groupby("user_id").sample(n=1, random_state=SEED)
print("Users available -> depressed:", len(one_dep), " control:", len(one_con))

# 4. pick 50 users from each class, shuffle
sample = pd.concat([
    one_dep.sample(n=N_PER_CLASS, random_state=SEED),
    one_con.sample(n=N_PER_CLASS, random_state=SEED),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample.insert(0, "id", [f"zh_{i+1:03d}" for i in range(len(sample))])
sample.insert(1, "language", "Chinese")
log_step(f"4. one post per user ({SELECTION}) + 50/50 sample", sample)


# (same method as Thai: cut at word boundaries, glue back with no spaces)

def split_into_thirds_zh(text):
    tokens = jieba.lcut(text)                                  # Chinese words (spaces kept as tokens)
    real = [i for i, tk in enumerate(tokens) if tk.strip() != ""]
    n = len(real)
    base, extra = n // 3, n % 3
    sizes = [base + (1 if i < extra else 0) for i in range(3)]
    cut1 = real[sizes[0]]
    cut2 = real[sizes[0] + sizes[1]]
    return ("".join(tokens[:cut1]).strip(),
            "".join(tokens[cut1:cut2]).strip(),
            "".join(tokens[cut2:]).strip())

sample[["prefix", "middle", "suffix"]] = sample["text"].apply(
    lambda t: pd.Series(split_into_thirds_zh(t)))
sample = sample.rename(columns={"text": "full"})

# safety check (ignoring spaces)
rebuilt = (sample["prefix"] + sample["middle"] + sample["suffix"]).str.replace(" ", "")
print("All splits correct:", (rebuilt == sample["full"].str.replace(" ", "")).all())

for part in ["prefix", "middle", "suffix"]:
    sample[f"kw_in_{part}"] = sample[part].str.contains("抑郁")
print("\nKeyword position (label 1 only):")
print(sample[sample["label"] == 1][["kw_in_prefix", "kw_in_middle", "kw_in_suffix"]].sum())

cols = ["id", "language", "user_id", "label", "full", "prefix", "middle", "suffix",
        "n_words", "has_keyword", "kw_in_prefix", "kw_in_middle", "kw_in_suffix",
        "self_reported", "suicidal_ideation"]
sample = sample[cols]

# saaving
sample.to_csv("chinese_sample_split.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(log).to_csv("chinese_cleaning_log.csv", index=False)
pool.drop(columns=["is_origin"]).to_csv("chinese_clean_full.csv", index=False, encoding="utf-8-sig")

print("\n", pd.DataFrame(log))
for f in ["chinese_sample_split.csv", "chinese_cleaning_log.csv",
          "chinese_clean_full.csv", "chinese_post_lengths.csv"]:
    print(f"{f}: {os.path.getsize(f) / 1e6:.1f} MB")

sample[["id", "label", "full", "prefix", "middle", "suffix"]].head(5)

Users available -> depressed: 292  control: 500
All splits correct: True

Keyword position (label 1 only):
kw_in_prefix    24
kw_in_middle    21
kw_in_suffix    11
dtype: int64

                                             step   posts  posts_label_1  \
0                 0. own posts (reposts removed)  146590          78725   
1                     1. clean text + drop empty  143507          77336   
2                             2. drop duplicates  140709          76283   
3                           3. keep 15-100 words   75869          42091   
4  4. one post per user (keyword) + 50/50 sample     100             50   

   posts_label_0  users_label_1  users_label_0  
0          67865            500            500  
1          66171            500            500  
2          64426            500            500  
3          33778            500            500  
4             50             50             50  
chinese_sample_split.csv: 0.0 MB
chinese_cleaning_log.csv: 0.0 MB
chinese_cl

,id,label,full,prefix,middle,suffix
0,zh_001,0,被朱正廷种草的口红必须拥有姓名！！ 1⃣ 嫩嫩的小番茄🍅 色！元气少女就是你！！ 2⃣ 高贵冷艳车厘子色🍒 ！气质小姐姐就是你！ 3⃣ 复古气场酒红色🍷 ！！霸气侧漏酷girl！！ [USER] [USER] 韶关,被朱正廷种草的口红必须拥有姓名！！ 1⃣ 嫩嫩的小番茄🍅 色！元气少女,就是你！！ 2⃣ 高贵冷艳车厘子色🍒 ！气质小姐姐就是你！ 3⃣ 复古,气场酒红色🍷 ！！霸气侧漏酷girl！！ [USER] [USER] 韶关
1,zh_002,0,被全家宠着的小姑娘很幸福了！ [心] 王诗龄爷爷会每年给王诗龄画一幅画，价值很不菲了！[加油][加油][加油],被全家宠着的小姑娘很幸福了！ [心],王诗龄爷爷会每年给王诗龄画一幅画，价值很,不菲了！[加油][加油][加油]
2,zh_003,0,In wonderland So we went on our way 地点：保国寺 摄影 宁波·浙江工商职业技术学院,In wonderland So we went on,our way 地点：保国寺 摄影,宁波·浙江工商职业技术学院
3,zh_004,1,当你觉得抑郁，难受，痛苦，想找个人倾诉，而别人觉得你只是矫情，无病呻吟，无理取闹的时候。 那种心情？？？,当你觉得抑郁，难受，痛苦，想找,个人倾诉，而别人觉得你只是矫情，无病呻吟,，无理取闹的时候。 那种心情？？？
4,zh_005,1,抑郁 这年头奥特曼都胖，所以我胖也不奇怪吧 [鼓掌],抑郁 这年头奥特曼都胖，,所以我胖也不,奇怪吧 [鼓掌]
